# Assessor de Seleção de Hiperparâmetros (LLM)
Roda entre rodadas de sweep. Auto-detecta a rodada pelo ledger.

**Provider (params.yaml > advisor):** primary = OpenAI/GPT — exporte `OPENAI_API_KEY`. Fallback automático = Ollama Cloud (`gemma4:31b`) se a chamada ao primary falhar; configure `advisor.fallback.base_url` com o endpoint do seu servidor Ollama.

In [ ]:
import sys, os
sys.path.append("..")   # para importar _advisor / _helpers
import _advisor, _helpers

CORPUS_ID = "folha"            # ou "tweets_bre2022"
MODEL = "lda"                  # "bertopic", "lda", "nmf" ou "stm"
BASE_OUTPUT = os.path.join("..", "data", "output", CORPUS_ID, MODEL)

In [ ]:
result = _advisor.run_advisor_round(CORPUS_ID, MODEL, BASE_OUTPUT)
print(f"Rodada: {result['round']}  |  fase: {result['phase']}  |  status: {result['status']}")
if result["invalid_keys"]:
    print("Chaves rejeitadas (fora do allowlist):", result["invalid_keys"])
print("Arquivos:", result["paths"])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["report_md"]))
print("\n--- params_patch proposto (aplicar manualmente ao params.yaml) ---")
import yaml
print(yaml.safe_dump(result["proposal"].get("params_patch", {}), allow_unicode=True, sort_keys=False))

## Fase site — auditoria de visualizações

Uma rodada por modelo × corpus (+ 1 da triangulação). Gera
`<run_dir>/advisor_viz/<slug>.md` para inclusão no site Quarto — prosa pura,
nunca toca `params.yaml`.

**Painel de duas leituras** (`params.yaml > advisor.viz_panel: true`): cada
rodada consulta os **dois** providers — primary (GPT) e o do bloco `fallback`
(gemma) — com o mesmo prompt e a mesma evidência, sem que um veja a resposta do
outro. As duas saem no mesmo arquivo, rotuladas `**Leitura A**` / `**Leitura B**`.
Consequências práticas:

- **custo e tempo dobram** por rodada (duas chamadas + eventual retry de cada);
- se **um** provider falhar, o arquivo sai com a leitura do outro (rodapé no
  singular) e o ledger marca `partial_panel`; só se os **dois** falharem a
  rodada levanta;
- slug sem CSV vira **um** placeholder, não dois — ele é inserido em código e
  sai idêntico dos dois providers;
- `> **Nota do autor ...` já presente num arquivo é **reinjetada** no re-run: a
  ressalva da auditoria humana não se perde ao regenerar o texto.

Desligue `viz_panel` para voltar ao comportamento antigo (uma leitura, com o
fallback entrando só se o primary falhar).


In [ ]:
import _advisor
from importlib import reload
reload(_advisor)

resultados = {}
for corpus in ["folha", "tweets_bre2022"]:
    for model in ["lda", "nmf", "stm", "bertopic"]:
        base = f"../data/output/{corpus}/{model}"
        r = _advisor.run_viz_audit(corpus, model, base)
        resultados[(corpus, model)] = r
        print(f"{corpus}/{model}: {len(r['written'])} viz | faltando: {r['missing'] or '—'}")

# 9a auditoria: triangulacao (evidencia = comparison_metrics.csv)
r = _advisor.run_viz_audit("all", "comparacao", "../data/output/comparison")
resultados[("all", "comparacao")] = r
print(f"comparacao: {len(r['written'])} viz | faltando: {r['missing'] or '—'}")
